In [23]:
from cuml.svm import SVC
import cupy as cp
%load_ext autoreload
%autoreload 2
import numpy as np
#import matplotlib.pyplot as plt
import os
#from tqdm import tqdm

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [26]:
def evalKernelSVM(base_dir,sig_name,sig_frac_train=0.8):
    bkg = np.load(f"{base_dir}/QCDBKG.npy")[:,2:4]
    sig = np.load(f"{base_dir}/{sig_name}.npy")[:,2:4]
    total = np.concatenate((bkg,sig),axis=0)
    lmin = np.min(total,axis=0)
    lmax = np.max(total,axis=0)
    sig = (sig-lmin)/(lmax-lmin)
    bkg = (bkg-lmin)/(lmax-lmin)
    ntrain = int(sig_frac_train*len(sig))
    sig_train, sig_test = sig[:ntrain], sig[ntrain:]
    bkg_train, bkg_test = bkg[:ntrain], bkg[ntrain:len(sig)]
    print(f"--Train on {ntrain} sig events, {ntrain} bkg events")

    train = np.concatenate((sig_train,bkg_train),axis=0)
    labels = np.concatenate((np.ones(sig_train.shape[0]),-1*np.ones(bkg_train.shape[0])),axis=0)
    shuf = np.random.permutation(len(train))
    train = train[shuf]
    labels = labels[shuf]
    svm = SVC(kernel='rbf',gamma='scale')
    svm = svm.fit(train,labels)

    step=0.05
    x,y = np.meshgrid(np.arange(-2,3+step,step),np.arange(-2,3+step,step))
    inp = np.stack((x,y)).reshape(2,-1).transpose(1,0)

    test = np.concatenate((sig_test,bkg_test),axis=0)
    test_labels = np.concatenate((np.ones(sig_test.shape[0]),-1*np.ones(bkg_test.shape[0])),axis=0)
    shuf = np.random.permutation(len(test))
    test = test[shuf]
    test_labels = test_labels[shuf]
    preds = svm.predict(test).values_host
    nTrue = np.count_nonzero(preds==test_labels)
    mean_acc = float(nTrue)/float(len(test_labels))
    
    return mean_acc

In [29]:
space="QCDBKG_mjjFlat-XYY_X3000_Y80_UL17-Wp3000_B400_UL17-Qstar2000_W400_UL17-Wkk_W3000_R400_UL17-RSG_M3000_UL17"
reductions=["Sum","Min","inv2norm","inv5norm","inv10norm"]
transforms=['None']
print(f"QUAK Space : {space}")
results = {}
for reduc in reductions:
    for transf in transforms:
        print(f"Running for transform {transf}, reduction {reduc}")
        base = f"quakSpaces/2017/transform-{transf}_reduce-{reduc}/{space}/"
        #signals = [k.split(".")[0] for k in os.listdir(base) if ".npy" in k and "QCDBKG" not in k]
        signals = ["XYY_X3000_Y80_UL17","Wp3000_B400_UL17","Qstar2000_W400_UL17","RSG_M3000_UL17","Wkk_W3000_R400_UL17"]
        accs = {}
        for s in signals:
            print(f"-{s}")
            mean_acc = evalKernelSVM(base,s)
            accs[s] = mean_acc
            print(f"--Acc {mean_acc:.5f}")
        results[f"({transf},{reduc})"] = accs

QUAK Space : QCDBKG_mjjFlat-XYY_X3000_Y80_UL17-Wp3000_B400_UL17-Qstar2000_W400_UL17-Wkk_W3000_R400_UL17-RSG_M3000_UL17
Running for transform None, reduction Sum
-XYY_X3000_Y80_UL17
--Train on 38896 sig events, 38896 bkg events
--Acc 0.93053
-Wp3000_B400_UL17
--Train on 34087 sig events, 34087 bkg events
--Acc 0.90172
-Qstar2000_W400_UL17
--Train on 28722 sig events, 28722 bkg events
--Acc 0.63703
-RSG_M3000_UL17
--Train on 34612 sig events, 34612 bkg events
--Acc 0.69560
-Wkk_W3000_R400_UL17
--Train on 28195 sig events, 28195 bkg events
--Acc 0.85374
Running for transform None, reduction Min
-XYY_X3000_Y80_UL17
--Train on 38896 sig events, 38896 bkg events
--Acc 0.89762
-Wp3000_B400_UL17
--Train on 34087 sig events, 34087 bkg events
--Acc 0.91622
-Qstar2000_W400_UL17
--Train on 28722 sig events, 28722 bkg events
--Acc 0.65081
-RSG_M3000_UL17
--Train on 34612 sig events, 34612 bkg events
--Acc 0.73304
-Wkk_W3000_R400_UL17
--Train on 28195 sig events, 28195 bkg events
--Acc 0.85026
Runni

/home/sambt/.local/lib/python3.7/site-packages/ipykernel_launcher.py:7: RuntimeWarning: invalid value encountered in true_divide
  import sys


--Train on 28195 sig events, 28195 bkg events


RuntimeError: Exception occured! file=/opt/anaconda3/conda-bld/libcuml_1600191077118/work/cpp/src/svm/smosolver.h line=262: SMO error: NaN found during fitting. This might be caused by floating point overflow. In such case using fp64 could help. Alternatively, try gamma='scale' kernel parameter.
Obtained 64 stack frames
#0 in /nobackup/users/sambt/anaconda3/envs/cuml/lib/python3.7/site-packages/cuml/common/../../../../libcuml++.so(+0x2159a8) [0x20002c2159a8]
#1 in /nobackup/users/sambt/anaconda3/envs/cuml/lib/python3.7/site-packages/cuml/common/../../../../libcuml++.so(+0x2167b4) [0x20002c2167b4]
#2 in /nobackup/users/sambt/anaconda3/envs/cuml/lib/python3.7/site-packages/cuml/common/../../../../libcuml++.so(_ZN2ML3SVM9SmoSolverIfE5SolveEPfiiS3_PS3_PiS4_PS5_S3_ii+0x2e20) [0x20002c4812e0]
#3 in /nobackup/users/sambt/anaconda3/envs/cuml/lib/python3.7/site-packages/cuml/common/../../../../libcuml++.so(_ZN2ML3SVM6svcFitIfEEvRKNS_10cumlHandleEPT_iiS6_RKNS0_12svmParameterERN8MLCommon6Matrix12

In [31]:
samps = list(results['(None,Sum)'].keys())
results2 = {s:{} for s in samps}
for s in samps:
    for k in results.keys():
        results2[s][k] = results[k][s]

In [38]:
import pandas as pd
df = pd.DataFrame(results2)

In [39]:
df

,XYY_X3000_Y80_UL17,Wp3000_B400_UL17,Qstar2000_W400_UL17,RSG_M3000_UL17,Wkk_W3000_R400_UL17
"(None,Min)",0.897624,0.916217,0.650815,0.733041,0.850262
"(None,Sum)",0.930533,0.901725,0.637028,0.695597,0.853738
"(None,inv2norm)",0.925905,0.913577,0.646358,0.726222,0.857001
"(None,inv5norm)",0.916341,0.918505,0.648865,0.733907,0.855724
